# Ordinal Encoding

In [36]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import OrdinalEncoder, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split, cross_val_score
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier

# Create sample dataset with ordinal features
np.random.seed(42)
n_samples = 500

import warnings
warnings.filterwarnings("ignore", category=UserWarning)

In [5]:
# Define ordinal categories
education_levels = ['High School', "Bachelor's", "Master's", 'PhD']
satisfaction_levels = ['Very Unsatisfied', 'Unsatisfied', 'Neutral', 'Satisfied', 'Very Satisfied']
income_brackets = ['Low', 'Medium', 'High', 'Very High']
sizes = ['XS', 'S', 'M', 'L', 'XL']

data = pd.DataFrame({
    'education': np.random.choice(education_levels, n_samples),
    'satisfaction': np.random.choice(satisfaction_levels, n_samples),
    'income_bracket': np.random.choice(income_brackets, n_samples),
    'shirt_size': np.random.choice(sizes, n_samples),
    'age': np.random.randint(22, 65, n_samples),
    'purchased': np.random.randint(0, 2, n_samples)
})

In [6]:
# Add some correlation
data.loc[data['education'] == 'PhD', 'purchased'] = 1
data.loc[data['satisfaction'] == 'Very Satisfied', 'purchased'] = 1
data['purchased'] = data['purchased'].clip(0, 1)

print("Sample Dataset with Ordinal Features:")
print(data.head(10))
print(f"\nDataset shape: {data.shape}")


Sample Dataset with Ordinal Features:
     education      satisfaction income_bracket shirt_size  age  purchased
0     Master's    Very Satisfied           High          S   38          1
1          PhD    Very Satisfied      Very High          M   33          1
2  High School  Very Unsatisfied            Low          M   53          1
3     Master's           Neutral           High          L   36          1
4     Master's       Unsatisfied            Low          L   37          1
5          PhD  Very Unsatisfied      Very High         XL   43          1
6  High School       Unsatisfied         Medium         XL   49          1
7  High School       Unsatisfied           High          S   41          0
8     Master's           Neutral            Low          S   54          0
9   Bachelor's       Unsatisfied            Low         XS   31          1

Dataset shape: (500, 6)


## Method 1: Basic OrdinalEncoder with Correct Ordering

In [7]:
print("\n" + "="*60)
print("Method 1: Proper Ordinal Encoding")
print("="*60)

# Define correct ordinal orderings
education_order = [['High School', "Bachelor's", "Master's", 'PhD']]
satisfaction_order = [['Very Unsatisfied', 'Unsatisfied', 'Neutral', 'Satisfied', 'Very Satisfied']]
income_order = [['Low', 'Medium', 'High', 'Very High']]
size_order = [['XS', 'S', 'M', 'L', 'XL']]

encoder = OrdinalEncoder(categories=education_order)
encoded_education = encoder.fit_transform(data[['education']])

print("\nEducation Encoding:")
print(f"Original categories: {education_order[0]}")
print(f"Encoded as: {[0, 1, 2, 3]}")
print(f"\nSample mappings:")
for i in range(5):
    print(f"  {data['education'].iloc[i]:15s} → {encoded_education[i][0]:.0f}")


Method 1: Proper Ordinal Encoding

Education Encoding:
Original categories: ['High School', "Bachelor's", "Master's", 'PhD']
Encoded as: [0, 1, 2, 3]

Sample mappings:
  Master's        → 2
  PhD             → 3
  High School     → 0
  Master's        → 2
  Master's        → 2


## Method 2: Wrong vs Right Ordering Comparison

In [8]:
print("\n" + "="*60)
print("Method 2: Impact of Incorrect Ordering")
print("="*60)

# Wrong: alphabetical ordering
wrong_encoder = OrdinalEncoder()  # Uses alphabetical by default
wrong_encoded = wrong_encoder.fit_transform(data[['education']])

print("\n❌ Alphabetical Ordering (WRONG):")
print(f"Categories in alphabetical order: {sorted(education_levels)}")
for cat in sorted(education_levels):
    idx = sorted(education_levels).index(cat)
    print(f"  {cat:15s} → {idx}")

print("\n✅ Correct Logical Ordering:")
for idx, cat in enumerate(education_order[0]):
    print(f"  {cat:15s} → {idx}")

print("\n⚠️ Problem: Alphabetical treats Bachelor's (0) as lowest education!")


Method 2: Impact of Incorrect Ordering

❌ Alphabetical Ordering (WRONG):
Categories in alphabetical order: ["Bachelor's", 'High School', "Master's", 'PhD']
  Bachelor's      → 0
  High School     → 1
  Master's        → 2
  PhD             → 3

✅ Correct Logical Ordering:
  High School     → 0
  Bachelor's      → 1
  Master's        → 2
  PhD             → 3

⚠️ Problem: Alphabetical treats Bachelor's (0) as lowest education!


## Method 3: Multiple Ordinal Features with ColumnTransformer

In [ ]:
print("\n" + "="*60)
print("Method 3: Multiple Ordinal Features")
print("="*60)

preprocessor = ColumnTransformer(
    transformers=[
        ('education', OrdinalEncoder(categories=education_order), ['education']),
        ('satisfaction', OrdinalEncoder(categories=satisfaction_order), ['satisfaction']),
        ('income', OrdinalEncoder(categories=income_order), ['income_bracket']),
        ('size', OrdinalEncoder(categories=size_order), ['shirt_size']),
        ('age', 'passthrough', ['age'])
    ]
)

X = data.drop('purchased', axis=1)
y = data['purchased']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42
)

X_train_encoded = preprocessor.fit_transform(X_train)

print(f"\nOriginal features: {X_train.columns.tolist()}")
print(f"Encoded shape: {X_train_encoded.shape}")
print(f"\nFirst row original:")
print(X_train.iloc[0])
print(f"\nFirst row encoded:")
print(X_train_encoded[0]) # pyright: ignore[reportIndexIssue]


Method 3: Multiple Ordinal Features

Original features: ['education', 'satisfaction', 'income_bracket', 'shirt_size', 'age']
Encoded shape: (375, 5)

First row original:
education             Bachelor's
satisfaction      Very Satisfied
income_bracket            Medium
shirt_size                    XL
age                           38
Name: 227, dtype: object

First row encoded:
[ 1.  4.  1.  4. 38.]


## Method 4: Model Performance with Ordinal Encoding

In [10]:
print("\n" + "="*60)
print("Method 4: Model Training with Ordinal Features")
print("="*60)

# Pipeline with ordinal encoding
pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', LogisticRegression(random_state=42, max_iter=1000))
])

pipeline.fit(X_train, y_train)

train_score = pipeline.score(X_train, y_train)
test_score = pipeline.score(X_test, y_test)

print(f"\nLogistic Regression with Ordinal Encoding:")
print(f"  Training Accuracy: {train_score:.4f}")
print(f"  Testing Accuracy: {test_score:.4f}")

# Get feature names after transformation
feature_names = (
    ['education', 'satisfaction', 'income_bracket', 'shirt_size', 'age']
)

# Get coefficients
coef = pipeline.named_steps['classifier'].coef_[0]
print(f"\nModel Coefficients (interpretable with ordinal features):")
for name, c in zip(feature_names, coef):
    print(f"  {name:20s}: {c:+.4f}")
    if c > 0:
        print(f"    → Each step up increases log-odds by {c:.4f}")
    else:
        print(f"    → Each step up decreases log-odds by {abs(c):.4f}")


Method 4: Model Training with Ordinal Features

Logistic Regression with Ordinal Encoding:
  Training Accuracy: 0.7653
  Testing Accuracy: 0.7280

Model Coefficients (interpretable with ordinal features):
  education           : +1.0227
    → Each step up increases log-odds by 1.0227
  satisfaction        : +0.6657
    → Each step up increases log-odds by 0.6657
  income_bracket      : -0.0327
    → Each step up decreases log-odds by 0.0327
  shirt_size          : +0.0967
    → Each step up increases log-odds by 0.0967
  age                 : -0.0161
    → Each step up decreases log-odds by 0.0161


## Method 5: Handling Unknown Categories

In [11]:
print("\n" + "="*60)
print("Method 5: Handling Unseen Categories")
print("="*60)

# Create encoder that handles unknowns
encoder_with_unknown = OrdinalEncoder(
    categories=education_order,
    handle_unknown='use_encoded_value',
    unknown_value=-1
)

# Fit on subset
train_education = pd.DataFrame({
    'education': ['High School', "Bachelor's", "Master's"]  # Missing PhD
})
encoder_with_unknown.fit(train_education)

# Test on full data
test_education = pd.DataFrame({
    'education': ['High School', 'PhD', "Bachelor's", 'Unknown Degree']
})

encoded = encoder_with_unknown.transform(test_education)
print("\nTraining categories: ['High School', 'Bachelor\\'s', 'Master\\'s']")
print("\nTest data encoding:")
for original, enc in zip(test_education['education'], encoded):
    status = "✅ Known" if enc[0] >= 0 else "⚠️ Unknown"
    print(f"  {original:20s} → {enc[0]:2.0f}  {status}")


Method 5: Handling Unseen Categories

Training categories: ['High School', 'Bachelor\'s', 'Master\'s']

Test data encoding:
  High School          →  0  ✅ Known
  PhD                  →  3  ✅ Known
  Bachelor's           →  1  ✅ Known
  Unknown Degree       → -1  ⚠️ Unknown


## Method 6: Custom Numerical Mapping (Alternative)

In [12]:
print("\n" + "="*60)
print("Method 6: Custom Numerical Mapping")
print("="*60)

# Sometimes equal spacing doesn't make sense
# Example: Years of education might be better than ordinal ranks
custom_mapping = {
    'High School': 12,
    "Bachelor's": 16,
    "Master's": 18,
    'PhD': 21
}

data['education_years'] = data['education'].map(custom_mapping)

print("\nComparing ordinal ranks vs custom mapping:")
print("\nOrdinal Encoding (equal spacing):")
print("  High School  → 0 (Δ=1)")
print("  Bachelor's   → 1 (Δ=1)")
print("  Master's     → 2 (Δ=1)")
print("  PhD          → 3")

print("\nCustom Mapping (years of education):")
print("  High School  → 12 (Δ=4)")
print("  Bachelor's   → 16 (Δ=2)")
print("  Master's     → 18 (Δ=3)")
print("  PhD          → 21")

print("\n💡 Custom mapping captures that Bachelor's takes 4 years,")
print("   while Master's typically takes 2 years.")


Method 6: Custom Numerical Mapping

Comparing ordinal ranks vs custom mapping:

Ordinal Encoding (equal spacing):
  High School  → 0 (Δ=1)
  Bachelor's   → 1 (Δ=1)
  Master's     → 2 (Δ=1)
  PhD          → 3

Custom Mapping (years of education):
  High School  → 12 (Δ=4)
  Bachelor's   → 16 (Δ=2)
  Master's     → 18 (Δ=3)
  PhD          → 21

💡 Custom mapping captures that Bachelor's takes 4 years,
   while Master's typically takes 2 years.


# Example 2: (Advanced) Comparing Encodings for Ordinal Data

In [14]:
# Ordinal feature with clear progression
satisfaction = np.random.choice(
    ['Very Unsatisfied', 'Unsatisfied', 'Neutral', 'Satisfied', 'Very Satisfied'],
    n_samples,
    p=[0.1, 0.15, 0.25, 0.35, 0.15]
)

# Target correlates with satisfaction (ordinal relationship)
satisfaction_scores = {
    'Very Unsatisfied': 0.1,
    'Unsatisfied': 0.3,
    'Neutral': 0.5,
    'Satisfied': 0.7,
    'Very Satisfied': 0.9
}

data = pd.DataFrame({
    'satisfaction': satisfaction,
    'age': np.random.randint(18, 70, n_samples),
    'income': np.random.randint(30000, 150000, n_samples)
})

# Create target with ordinal relationship
data['purchased'] = [
    np.random.random() < satisfaction_scores[sat] 
    for sat in data['satisfaction']
]
data['purchased'] = data['purchased'].astype(int)

In [15]:
print("Comparing Encoding Strategies for Ordinal Data")
print("="*60)
print(f"\nDataset: {n_samples} samples")
print(f"Target (purchased) has clear ordinal relationship with satisfaction")

# Define ordinal order
satisfaction_order = [['Very Unsatisfied', 'Unsatisfied', 'Neutral', 'Satisfied', 'Very Satisfied']]

X = data.drop('purchased', axis=1)
y = data['purchased']


Comparing Encoding Strategies for Ordinal Data

Dataset: 500 samples
Target (purchased) has clear ordinal relationship with satisfaction


In [19]:
# Strategy 1: Ordinal Encoding
preprocessor_ordinal = ColumnTransformer([
    ('satisfaction', OrdinalEncoder(categories=satisfaction_order), ['satisfaction']),
    ('numeric', 'passthrough', ['age', 'income'])
])

# Strategy 2: One-Hot Encoding
preprocessor_ohe = ColumnTransformer([
    ('satisfaction', OneHotEncoder(drop='first', sparse_output=False), ['satisfaction']),
    ('numeric', 'passthrough', ['age', 'income'])
])

# Strategy 3: Wrong Ordinal (alphabetical)
preprocessor_wrong = ColumnTransformer([
    ('satisfaction', OrdinalEncoder(), ['satisfaction']),  # Alphabetical!
    ('numeric', 'passthrough', ['age', 'income'])
])


In [22]:

strategies = {
    'Ordinal (Correct Order)': preprocessor_ordinal,
    'One-Hot Encoding': preprocessor_ohe,
    'Ordinal (Alphabetical - WRONG)': preprocessor_wrong
}

algorithms = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42),
    'Gradient Boosting': GradientBoostingClassifier(n_estimators=100, random_state=42)
}


In [23]:
results = []

print("\nCross-Validation Results:")
print("-" * 60)

for strategy_name, preprocessor in strategies.items():
    print(f"\n{strategy_name}:")
    
    for algo_name, algorithm in algorithms.items():
        pipeline = Pipeline([
            ('preprocessor', preprocessor),
            ('classifier', algorithm)
        ])
        
        scores = cross_val_score(pipeline, X, y, cv=5, scoring='accuracy')
        mean_score = scores.mean()
        std_score = scores.std()
        
        results.append({
            'Strategy': strategy_name,
            'Algorithm': algo_name,
            'Mean_Score': mean_score,
            'Std_Score': std_score
        })
        
        print(f"  {algo_name:20s}: {mean_score:.4f} (+/- {std_score:.4f})")

results_df = pd.DataFrame(results)


Cross-Validation Results:
------------------------------------------------------------

Ordinal (Correct Order):
  Logistic Regression : 0.7060 (+/- 0.0224)
  Random Forest       : 0.6580 (+/- 0.0417)
  Gradient Boosting   : 0.6700 (+/- 0.0283)

One-Hot Encoding:
  Logistic Regression : 0.6940 (+/- 0.0150)
  Random Forest       : 0.6660 (+/- 0.0196)
  Gradient Boosting   : 0.6840 (+/- 0.0265)

Ordinal (Alphabetical - WRONG):
  Logistic Regression : 0.5760 (+/- 0.0361)
  Random Forest       : 0.6660 (+/- 0.0361)
  Gradient Boosting   : 0.6680 (+/- 0.0248)


In [27]:
print("\n" + "="*60)
print("Key Findings:")
print("="*60)

# Find best per algorithm
best_strategies = results_df.loc[results_df.groupby('Algorithm')['Mean_Score'].idxmax()]

print("\nBest Strategy per Algorithm:")
for _, row in best_strategies.iterrows():
    print(f"  {row['Algorithm']:20s}: {row['Strategy']:35s} ({row['Mean_Score']:.4f})")


Key Findings:

Best Strategy per Algorithm:
  Gradient Boosting   : One-Hot Encoding                    (0.6840)
  Logistic Regression : Ordinal (Correct Order)             (0.7060)
  Random Forest       : Ordinal (Alphabetical - WRONG)      (0.6660)


### Key Insights

1. Ordinal Encoding (Correct Order):
- ✅ Captures the natural progression in satisfaction
- ✅ Single feature reduces dimensionality
- ✅ Works well with all algorithms
- ✅ Most interpretable for ordinal data

2. One-Hot Encoding:
- ⚠️ Creates 4 binary features (drops first level)
- ⚠️ Doesn't explicitly capture ordering
- ✓ Still works but less efficient
- ✓ May work better if ordinal relationship is weak

3. Wrong Ordinal (Alphabetical):
- ❌ Incorrect ordering: Neutral < Satisfied < Unsatisfied < Very Satisfied < Very Unsatisfied
- ❌ Breaks the natural progression
- ❌ Can significantly hurt performance
- ❌ Demonstrates importance of specifying correct order

# Example 3: Survey Data with Multiple Ordinal Scales

In [29]:
# Define all ordinal scales
n_respondents = 500
likert_5 = ['Strongly Disagree', 'Disagree', 'Neutral', 'Agree', 'Strongly Agree']
frequency = ['Never', 'Rarely', 'Sometimes', 'Often', 'Always']
satisfaction = ['Very Unsatisfied', 'Unsatisfied', 'Neutral', 'Satisfied', 'Very Satisfied']
priority = ['Not Important', 'Slightly Important', 'Moderately Important', 
            'Very Important', 'Extremely Important']

survey_data = pd.DataFrame({
    'product_quality': np.random.choice(likert_5, n_respondents),
    'usage_frequency': np.random.choice(frequency, n_respondents),
    'overall_satisfaction': np.random.choice(satisfaction, n_respondents),
    'feature_importance': np.random.choice(priority, n_respondents),
    'age_group': np.random.choice(['18-25', '26-35', '36-45', '46-55', '56+'], n_respondents),
    'spending': np.random.randint(100, 5000, n_respondents)
})

In [30]:
print("Customer Survey Analysis with Ordinal Encoding")
print("="*60)
print("\nSurvey Data Sample:")
print(survey_data.head(10))

# Define all ordinal orderings
ordinal_features = {
    'product_quality': [likert_5],
    'usage_frequency': [frequency],
    'overall_satisfaction': [satisfaction],
    'feature_importance': [priority],
    'age_group': [['18-25', '26-35', '36-45', '46-55', '56+']]
}


Customer Survey Analysis with Ordinal Encoding

Survey Data Sample:
  product_quality usage_frequency overall_satisfaction    feature_importance  \
0           Agree           Often            Satisfied        Very Important   
1  Strongly Agree           Never            Satisfied    Slightly Important   
2         Neutral          Always            Satisfied        Very Important   
3  Strongly Agree       Sometimes            Satisfied  Moderately Important   
4  Strongly Agree       Sometimes       Very Satisfied         Not Important   
5        Disagree           Never       Very Satisfied        Very Important   
6         Neutral           Often          Unsatisfied         Not Important   
7         Neutral           Often     Very Unsatisfied    Slightly Important   
8         Neutral          Always            Satisfied         Not Important   
9  Strongly Agree           Never          Unsatisfied    Slightly Important   

  age_group  spending  
0     46-55      2103  
1  

In [33]:
print("\n" + "="*60)
print("Ordinal Feature Mappings:")
print("="*60)

for feature, categories in ordinal_features.items():
    print(f"\n{feature}:")
    for idx, category in enumerate(categories[0]):
        print(f"  {idx} ← {category}")

# Create preprocessor
transformers = []
for feature, categories in ordinal_features.items():
    transformers.append((
        feature,
        OrdinalEncoder(categories=categories),
        [feature]
    ))


Ordinal Feature Mappings:

product_quality:
  0 ← Strongly Disagree
  1 ← Disagree
  2 ← Neutral
  3 ← Agree
  4 ← Strongly Agree

usage_frequency:
  0 ← Never
  1 ← Rarely
  2 ← Sometimes
  3 ← Often
  4 ← Always

overall_satisfaction:
  0 ← Very Unsatisfied
  1 ← Unsatisfied
  2 ← Neutral
  3 ← Satisfied
  4 ← Very Satisfied

feature_importance:
  0 ← Not Important
  1 ← Slightly Important
  2 ← Moderately Important
  3 ← Very Important
  4 ← Extremely Important

age_group:
  0 ← 18-25
  1 ← 26-35
  2 ← 36-45
  3 ← 46-55
  4 ← 56+


In [ ]:
preprocessor = ColumnTransformer(transformers=transformers)

# Prepare data
X = survey_data.drop('spending', axis=1)
y = survey_data['spending']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Train model
pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('regressor', LinearRegression())
])

pipeline.fit(X_train, y_train)

,steps,"[('preprocessor', ...), ('regressor', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('product_quality', ...), ('usage_frequency', ...), ...]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


In [38]:
train_score = pipeline.score(X_train, y_train)
test_score = pipeline.score(X_test, y_test)

print("\n" + "="*60)
print("Model Performance:")
print("="*60)
print(f"\nLinear Regression with Ordinal Features:")
print(f"  Training R²: {train_score:.4f}")
print(f"  Testing R²:  {test_score:.4f}")


Model Performance:

Linear Regression with Ordinal Features:
  Training R²: 0.0190
  Testing R²:  -0.0261


In [39]:
# Interpret coefficients
feature_names = list(ordinal_features.keys())
coefficients = pipeline.named_steps['regressor'].coef_

print("\n" + "="*60)
print("Feature Importance (Coefficients):")
print("="*60)
print("\nInterpretation: Effect of moving up one ordinal level")

for name, coef in sorted(zip(feature_names, coefficients), key=lambda x: abs(x[1]), reverse=True):
    print(f"\n{name}:")
    print(f"  Coefficient: {coef:+.2f}")
    if abs(coef) > 50:
        impact = "Strong"
    elif abs(coef) > 20:
        impact = "Moderate"
    else:
        impact = "Weak"
    
    direction = "increases" if coef > 0 else "decreases"
    print(f"  Impact: {impact} - Each level up {direction} spending by ${abs(coef):.2f}")


Feature Importance (Coefficients):

Interpretation: Effect of moving up one ordinal level

age_group:
  Coefficient: -130.56
  Impact: Strong - Each level up decreases spending by $130.56

usage_frequency:
  Coefficient: -34.26
  Impact: Moderate - Each level up decreases spending by $34.26

product_quality:
  Coefficient: -25.31
  Impact: Moderate - Each level up decreases spending by $25.31

overall_satisfaction:
  Coefficient: +22.28
  Impact: Moderate - Each level up increases spending by $22.28

feature_importance:
  Coefficient: +8.98
  Impact: Weak - Each level up increases spending by $8.98
